# 개요
파인튜닝 평가


5번 코드에서 생성한 테스트 데이터 평가하는 코드

```
- 비대상 테스트 데이터 : data1B, data_3B, data_8B, data_8B_Alpha
Llama-3.2-1B-Instruct_with_exec.zip
Llama-3.2-3B-Instruct_with_exec.zip
Llama-3.1-8B-Instruct_with_exec.zip
Llama-3-Alpha-Ko-8B-Instruct_with_exec.zip

너희는 나의 실수다.

- 대상 테스트 데이터 : data_1B_2, data_3B_2, data_8B_ce, data_8B_Alpha_2
Llama-3.2-1B-Instruct_with_exec2.zip
Llama-3.2-3B-Instruct_with_exec2.zip
Llama-3.1-8B-Instruct-text-to-sql-FT-olist-config-edit_with_exec.zip
`
테스트 데이터 제작 시 SQL DB 방언 sqlite 변환에 GPT-5.4 모델 사용.
configuration 재설정으로 학습률/학습 시간 증가.
```

#0. 환경설정

In [104]:
pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 6.2 MB/s eta 0:00:00


In [105]:
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from tqdm import tqdm

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

#1.데이터 불러오기

In [1]:
!wget https://github.com/leejunho12316/LLM_FineTuning_2/raw/f621ee0a362638d49bc94cd4f3644e545db8e876/5.FineTuning_result/Llama-3.2-1B-Instruct/Llama-3.2-1B-Instruct_with_exec.zip
!wget https://github.com/leejunho12316/LLM_FineTuning_2/raw/e521c1be8768bbc4bd65d09da1d1627d5a6add32/5.FineTuning_result/Llama-3.1-8B-Instruct/Llama-3.1-8B-Instruct_with_exec.zip
!wget https://github.com/leejunho12316/LLM_FineTuning_2/raw/e521c1be8768bbc4bd65d09da1d1627d5a6add32/5.FineTuning_result/Llama-3.2-3B-Instruct/Llama-3.2-3B-Instruct_with_exec.zip
!wget https://github.com/leejunho12316/LLM_FineTuning_2/raw/6715758fb5c54ec97fbb967c6f6390ba998b0ba0/5.FineTuning_result/Llama-3-Alpha-Ko-8B-Instruct/Llama-3-Alpha-Ko-8B-Instruct_with_exec.zip
!wget https://github.com/leejunho12316/LLM_FineTuning_2/raw/8671e8d5446cdf831861d82603614d04d68681ba/5.FineTuning_result/Llama-3.1-8B-Instruct-config-edit/Llama-3.1-8B-Instruct-text-to-sql-FT-olist-config-edit_with_exec.zip
!wget https://github.com/leejunho12316/LLM_FineTuning_2/raw/22231b0c8f89b6b34f2d91537424c102ea510a97/5.FineTuning_result/Llama-3.2-1B-Instruct/Llama-3.2-1B-Instruct_with_exec2.zip
!wget https://github.com/leejunho12316/LLM_FineTuning_2/raw/22231b0c8f89b6b34f2d91537424c102ea510a97/5.FineTuning_result/Llama-3.2-3B-Instruct/Llama-3.2-3B-Instruct_with_exec2.zip
!wget https://github.com/leejunho12316/LLM_FineTuning_2/raw/7e5fce5c41599c42f4f19d396e43f71976adfead/5.FineTuning_result/Llama-3-Alpha-Ko-8B-Instruct/Llama-3-Alpha-Ko-8B-Instruct_with_exec2.zip

--2026-06-28 01:26:59--  https://github.com/leejunho12316/LLM_FineTuning_2/raw/f621ee0a362638d49bc94cd4f3644e545db8e876/5.FineTuning_result/Llama-3.2-1B-Instruct/Llama-3.2-1B-Instruct_with_exec.zip
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/leejunho12316/LLM_FineTuning_2/f621ee0a362638d49bc94cd4f3644e545db8e876/5.FineTuning_result/Llama-3.2-1B-Instruct/Llama-3.2-1B-Instruct_with_exec.zip [following]
--2026-06-28 01:26:59--  https://raw.githubusercontent.com/leejunho12316/LLM_FineTuning_2/f621ee0a362638d49bc94cd4f3644e545db8e876/5.FineTuning_result/Llama-3.2-1B-Instruct/Llama-3.2-1B-Instruct_with_exec.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... con

In [187]:
import zipfile

zip_file_path_whole = ['Llama-3.2-1B-Instruct_with_exec.zip', 'Llama-3.2-3B-Instruct_with_exec.zip',
                       'Llama-3.1-8B-Instruct_with_exec.zip', 'Llama-3-Alpha-Ko-8B-Instruct_with_exec.zip',
                       'Llama-3.2-1B-Instruct_with_exec2.zip', 'Llama-3.2-3B-Instruct_with_exec2.zip',
                       'Llama-3.1-8B-Instruct-text-to-sql-FT-olist-config-edit_with_exec.zip', 'Llama-3-Alpha-Ko-8B-Instruct_with_exec2.zip']
for zip_file_path in zip_file_path_whole:
  with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
      zip_ref.extractall('.')
  print(f"Extracted all contents of {zip_file_path} to current directory.")

Extracted all contents of Llama-3.2-1B-Instruct_with_exec.zip to current directory.
Extracted all contents of Llama-3.2-3B-Instruct_with_exec.zip to current directory.
Extracted all contents of Llama-3.1-8B-Instruct_with_exec.zip to current directory.
Extracted all contents of Llama-3-Alpha-Ko-8B-Instruct_with_exec.zip to current directory.
Extracted all contents of Llama-3.2-1B-Instruct_with_exec2.zip to current directory.
Extracted all contents of Llama-3.2-3B-Instruct_with_exec2.zip to current directory.
Extracted all contents of Llama-3.1-8B-Instruct-text-to-sql-FT-olist-config-edit_with_exec.zip to current directory.
Extracted all contents of Llama-3-Alpha-Ko-8B-Instruct_with_exec2.zip to current directory.


In [188]:
import pandas as pd

data_1B = pd.read_csv('Llama-3.2-1B-Instruct_with_exec.csv')
data_1B_2 = pd.read_csv('Llama-3.2-1B-Instruct_with_exec2.csv')
data_3B = pd.read_csv('Llama-3.2-3B-Instruct_with_exec.csv')
data_3B_2 = pd.read_csv('Llama-3.2-3B-Instruct_with_exec2.csv')
data_8B = pd.read_csv('Llama-3.1-8B-Instruct_with_exec.csv')
data_8B_ce = pd.read_csv('Llama-3.1-8B-Instruct-text-to-sql-FT-olist-config-edit_with_exec.csv')
data_8B_Alpha = pd.read_csv('Llama-3-Alpha-Ko-8B-Instruct_with_exec.csv')
data_8B_Alpha2 = pd.read_csv('Llama-3-Alpha-Ko-8B-Instruct_with_exec2.csv')

#2. 결과 분석

```
error 종류
1. success : 성공
2. success(converted) : sqlite 형식으로 변환해서 성공
3. error : SQL too long (** chars) : 비정상적으로 긴 SQL
4. error: Execution failed on sql <SQL문> : SQL문 실행 오류
5. error: wrong format (no '쿼리 작성:' prefix)
```

In [189]:
import pandas as pd

def categorize_status(status: str) -> str:
    """response_status를 5가지 카테고리로 분류"""
    s = str(status).strip()

    if s == "success":
        return "1. success"
    elif s == "success(converted)":
        return "2. success(converted)"
    elif "SQL too long" in s:
        return "3. SQL too long"
    elif "wrong format" in s:
        return "5. wrong format"
    elif s.startswith("error"):
        return "4. SQL execution error"
    else:
        return "6. other"

for data,name in [(data_1B, 'data_1B'), (data_1B_2, 'data_1B_2'), (data_3B, 'data_3B'), (data_3B_2, 'data_3B_2'),
                  (data_8B, 'data_8B'), (data_8B_ce, 'data_8B_ce'), (data_8B_Alpha, 'data_8B_Alpha'), (data_8B_Alpha2, 'data_8B_Alpha2')]:

  counts = data['response_status'].apply(categorize_status).value_counts().sort_index()

  print(f"\n=== {name} response_status 비율 ===")
  print(f"\n비율 (%):")
  print((counts / len(data) * 100).round(1))




=== data_1B response_status 비율 ===

비율 (%):
response_status
1. success                73.0
2. success(converted)     18.0
3. SQL too long            1.0
4. SQL execution error     8.0
Name: count, dtype: float64

=== data_1B_2 response_status 비율 ===

비율 (%):
response_status
1. success                73.0
2. success(converted)     18.7
3. SQL too long            1.0
4. SQL execution error     7.3
Name: count, dtype: float64

=== data_3B response_status 비율 ===

비율 (%):
response_status
1. success                70.4
2. success(converted)     23.5
4. SQL execution error     6.1
Name: count, dtype: float64

=== data_3B_2 response_status 비율 ===

비율 (%):
response_status
1. success                70.4
2. success(converted)     23.1
4. SQL execution error     6.5
Name: count, dtype: float64

=== data_8B response_status 비율 ===

비율 (%):
response_status
1. success                34.4
2. success(converted)      8.6
4. SQL execution error     2.7
5. wrong format           54.3
Name: count, dtype: f

In [96]:
for i in range(0,10):
  print(data_1B['response_status'][data_1B['response_status'].str.contains('Execution failed on sql')].iloc[i])
  print('--------')

error: Execution failed on sql 'SELECT
  p.product_category_name,
  AVG(i.price) AS avg_price,
  SUM(i.quantity) AS total_quantity
FROM order_items i
JOIN products p ON p.product_id = i.product_id
WHERE LENG
--------
error: Execution failed on sql 'SELECT
  i.seller_id,
  COUNT(DISTINCT i.product_id) AS distinct_products,
  AVG(i.price) AS avg_price,
  AVG(i.price * i.product_photos_qty) AS avg_revenue
FROM order_items i
--------
error: Execution failed on sql 'SELECT
  c.customer_unique_id,
  COUNT(DISTINCT g.geolocation_state) AS distinct_state_count
FROM customers c
JOIN (
  SELECT
    geolocation_zip_code_prefix
  FROM geolocatio
--------
error: Execution failed on sql 'SELECT
  COUNT(DISTINCT c.customer_id) AS distinct_customers,
  AVG(g.geolocation_lat) AS avg_latitude
FROM customers c
JOIN (
  SELECT
    geolocation_zip_code_prefix,
    MA
--------
error: Execution failed on sql 'SELECT
  AVG(freight_value) AS avg_freight,
  CASE
    WHEN strftime('%w', shipping_limit_date) = '0

In [100]:
data_8B_Alpha

,index,prompt,response,label,response_tables,response_exec_result,response_status,label_tables,label_exec_result,label_status,match
0,0,<|begin_of_text|><|start_header_id|>system<|en...,"쿼리 작성: SELECT\n SUBSTRING(seller_city, 1, 1) ...","\n쿼리 작성: SELECT LOWER(SUBSTRING(seller_city, 1...",sellers,"city_first_char,seller_count\ns,1072\nc,365\nb...",success,sellers,"initial,seller_count\ns,1072\nc,365\nb,233\nr,...",success,False
1,1,<|begin_of_text|><|start_header_id|>system<|en...,쿼리 작성: SELECT product_id\nFROM products\nWHERE...,\n쿼리 작성: SELECT product_id\nFROM products\nWHE...,products,product_id\n005030ef108f58b46b78116f754d8d38\n...,success,products,product_id\n005030ef108f58b46b78116f754d8d38\n...,success,True
2,2,<|begin_of_text|><|start_header_id|>system<|en...,"쿼리 작성: SELECT\n c.customer_city,\n AVG(TIMES...","\n쿼리 작성: SELECT\n c.customer_city,\n AVG(TIM...","customers,orders","customer_city,avg_hours_to_carrier\nanhembi,48...",success(converted),"customers,orders","customer_city,avg_hours_to_carrier\nonda verde...",success(converted),False
3,3,<|begin_of_text|><|start_header_id|>system<|en...,"쿼리 작성: SELECT\n p.product_category_name,\n A...","\n쿼리 작성: SELECT p.product_category_name,\n ...","order_items,products","product_category_name,avg_price,item_count\npc...",success,"order_items,products","product_category_name,avg_price,item_count\npc...",success,False
4,4,<|begin_of_text|><|start_header_id|>system<|en...,쿼리 작성: SELECT\n AVG(\n CASE\n WHEN (r...,\n쿼리 작성: SELECT\n 1.0 * SUM(CASE WHEN COALESC...,order_reviews,pct_title_or_message_with_exclamation\n0.05997...,success(converted),order_reviews,ratio_with_exclamation\n0.05997540917519955\n,success,False
...,...,...,...,...,...,...,...,...,...,...,...
472,472,<|begin_of_text|><|start_header_id|>system<|en...,"쿼리 작성: SELECT\n p.product_category_name,\n A...","\n쿼리 작성: SELECT\n p.product_category_name,\n ...","order_items,products","product_category_name,avg_photos,avg_price\npc...",success,"order_items,products","product_category_name,avg_photos,avg_price\nci...",success,False
473,473,<|begin_of_text|><|start_header_id|>system<|en...,"쿼리 작성: SELECT\n c.customer_state,\n SUM(CASE...",\n쿼리 작성: SELECT\n YEAR(o.order_purchase_times...,"customers,orders","customer_state,delivered_count,undelivered_cou...",success,"customers,orders","year,customer_state,total_orders,delivered_rat...",success(converted),False
474,474,<|begin_of_text|><|start_header_id|>system<|en...,쿼리 작성: SELECT\n DATE(shipping_limit_date) AS ...,\n쿼리 작성: SELECT DATE(shipping_limit_date) AS d...,order_items,"date,total_freight\n2018-05-01,1166.4\n2018-05...",success,order_items,"date,total_freight\n2018-05-01,1166.4\n2018-05...",success,True
475,475,<|begin_of_text|><|start_header_id|>system<|en...,"쿼리 작성: SELECT\n s.seller_state,\n SUM(oi.fre...","\n쿼리 작성: SELECT\n s.seller_state,\n SUM(oi.f...","order_items,sellers","seller_state,total_freight,total_price,freight...",success,"order_items,sellers","seller_state,total_freight,total_price,freight...",success,False


In [101]:
for i in range(0,10):
  print(data_8B_Alpha[][data_8B_Alpha['response_status'].str.contains('Execution failed on sql')].iloc[i])
  print('--------')

쿼리 작성: SELECT
  payment_type
FROM order_payments
GROUP BY payment_type
HAVING SUM(CASE WHEN payment_type IN (
  SELECT payment_type
  FROM order_payments
  GROUP BY payment_type
  HAVING COUNT(DISTINCT payment_type) = 1
) THEN 1 ELSE 0
) = 0
ORDER BY payment_type;
--------
쿼리 작성: SELECT
  p.payment_type,
  COUNT(*) AS order_count
FROM orders o
JOIN (
  SELECT
    order_id,
    SUM(payment_value) AS total_payment
  FROM order_payments
  GROUP BY order_id
) p ON p.order_id = o.order_id
WHERE o.order_purchase_timestamp >= '2017-07-01'
  AND o.order_purchase_timestamp < '2017-10-01'
  AND p.total_payment >= 250
  AND DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp) <= 10
GROUP BY p.payment_type
ORDER BY p.payment_type;
--------
쿼리 작성: SELECT
  AVG(freight_sum / COUNT(*)) AS avg_freight_per_order
FROM (
  SELECT
    order_id,
    SUM(freight_value) AS freight_sum
  FROM order_items
  WHERE shipping_limit_date >= '2017-01-01' AND shipping_limit_date < '2018-01-01'
  GROU

#3.LLM 사용 준비

In [53]:
import re
#Llama chat template 프롬프트에서 query와 ddl 추출.
def parse_prompt(text: str) -> tuple[str, str]:

    query_match = re.search(
        r"입력 텍스트:\s*(.+?)(?=\s*\n+\s*DDL statements:|\s*$)",
        text,
        flags=re.DOTALL,
    )
    query = query_match.group(1).strip() if query_match else ""

    ddl_match = re.search(
        r"DDL statements:\s*(.+?)(?=\s*\n+\s*위의 테이블 명세)",
        text,
        flags=re.DOTALL,
    )
    ddl_statement = ddl_match.group(1).strip() if ddl_match else ""

    return query, ddl_statement

In [107]:
data_1B.head(3)

,index,prompt,response,label,response_tables,response_exec_result,response_status,label_tables,label_exec_result,label_status,match
0,0,<|begin_of_text|><|start_header_id|>system<|en...,"쿼리 작성: SELECT seller_city, COUNT(*) AS seller_...","\n쿼리 작성: SELECT LOWER(SUBSTRING(seller_city, 1...",sellers,"seller_city,seller_count\nsao paulo,694\ncurit...",success,sellers,"initial,seller_count\ns,1072\nc,365\nb,233\nr,...",success,False
1,1,<|begin_of_text|><|start_header_id|>system<|en...,쿼리 작성: SELECT product_id\nFROM products\nWHERE...,\n쿼리 작성: SELECT product_id\nFROM products\nWHE...,products,product_id\n3bb7f144022e6732727d8d838a7b13b3\n...,success,products,product_id\n005030ef108f58b46b78116f754d8d38\n...,success,False
2,2,<|begin_of_text|><|start_header_id|>system<|en...,"쿼리 작성: SELECT\n c.customer_city,\n AVG(TIMES...","\n쿼리 작성: SELECT\n c.customer_city,\n AVG(TIM...","customers,orders","customer_city,avg_hours_to_carrier\nsarapui,61...",success(converted),"customers,orders","customer_city,avg_hours_to_carrier\nonda verde...",success(converted),False


#4. LLM-as-a-Judge

##4-1. response quality

In [154]:
import pandas as pd
import re
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from tqdm import tqdm

# ===== 프롬프트 정의 (전역) =====
SYSTEM_PROMPT = """
#역할
당신은 Text-to-SQL FineTuning 평가자입니다.
입력된 정보들을 보고 LLM이 올바르게 Fine Tuning 되었는지 평가해주세요.

prompt는 입력된 프롬프트, ddl_statement는 사용된 DB의 DDL문과 예시 값입니다.
response는 Fine-Tuning된 모델의 응답, label은 모범 정답입니다.

#평가 기준
1. response가 prompt의 요구사항을 충족하는지
2. response가 label과 동일한 의도/결과를 제공하는지
3. ddl statement를 보았을 때 response가 존재하는 컬럼/테이블을 참조하는지
4. response의 SQL 형식/문법이 올바른지
5. response가 prompt에서 요구하는 조건을 충족하는지 (LIMIT, ORDER BY, DISTINCT 등)

#출력 형식
평가 기준 별 만족 여부와 그 이유를 적고 만족한 평가 기준 개수로 총점을 매겨주세요.총점은 무조건 정수 하나로만 대답하세요
예시)
[평가]
1 - 만족. 이유.
2 - 불만족. 이유.
3 - 만족. 이유.
...
[총점]
4
"""

HUMAN_PROMPT = """
prompt : {prompt}
ddl_statement : {ddl_statement}
response : {response}
label : {label}
"""


def extract_score(evaluation: str) -> int | None:
    """LLM 평가 결과 텍스트에서 [총점] 뒤의 정수를 추출."""
    if not isinstance(evaluation, str):
        return None
    # [총점] 다음에 오는 첫 정수 매칭
    match = re.search(r"\[총점\]\s*(\d+)", evaluation)
    return int(match.group(1)) if match else None


def evaluate_with_llm(df: pd.DataFrame, output_csv: str, model: str = "gpt-4o-mini") -> pd.DataFrame:
    """
    LLM으로 Fine-Tuning 결과를 평가하고 CSV로 저장.

    Args:
        df:         평가할 DataFrame (prompt, response, label 컬럼 필요)
        output_csv: 결과 저장 경로
        model:      사용할 OpenAI 모델 (기본: gpt-4o-mini)

    Returns:
        평가 결과가 담긴 DataFrame
    """
    # 1. LLM 체인 구성
    llm = ChatOpenAI(model=model, temperature=0)
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human",  HUMAN_PROMPT),
    ])
    chain = prompt_template | llm | StrOutputParser()

    print(f"평가 대상: {len(df)}개")

    # 2. 평가 루프
    evaluations = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="LLM 평가"):
        prompt_text, ddl_statement = parse_prompt(row['prompt'])

        try:
            evaluation = chain.invoke({
                "prompt":        prompt_text,
                "ddl_statement": ddl_statement,
                "response":      row['response'],
                "label":         row['label'],
            })
        except Exception as e:
            evaluation = f"[ERROR] {e}"

        # 총점 추출
        score = extract_score(evaluation)

        evaluations.append({
            "index":      row['index'],
            "prompt":     prompt_text,
            "ddl":        ddl_statement,
            "response":   row['response'],
            "label":      row['label'],
            "evaluation": evaluation,
            "score":      score,            # ← 새로 추가된 컬럼
        })

    # 3. 결과 저장
    df_result = pd.DataFrame(evaluations)
    df_result.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\n완료: {output_csv} ({len(df_result)}개 평가 완료)")

    # 4. 간단 요약
    valid_scores = df_result['score'].dropna()
    if len(valid_scores) > 0:
        print(f"\n=== 점수 통계 ===")
        print(f"평균:   {valid_scores.mean():.2f}")
        print(f"중앙값: {valid_scores.median():.1f}")
        print(f"분포:")
        print(valid_scores.value_counts().sort_index())

    return df_result

In [155]:
os.makedirs("LLM-as-a-Judge", exist_ok=True)

# result_1b = evaluate_with_llm(data_1B[:10], "Llama-3.2-1B-Instruct_llm_eval.csv")
result_8b = evaluate_with_llm(data_8B[:10], "LLM-as-a-Judge/Llama-3.2-8B-Instruct_llm_eval.csv",
                              model = "gpt-5.4-nano")


평가 대상: 10개


LLM 평가: 100%|██████████| 10/10 [00:28<00:00,  2.86s/it]


완료: LLM-as-a-Judge/Llama-3.2-8B-Instruct_llm_eval.csv (10개 평가 완료)

=== 점수 통계 ===
평균:   3.30
중앙값: 3.5
분포:
score
2    3
3    2
4    4
5    1
Name: count, dtype: int64


##4-2. response & label match 여부
단순 일치로는 평가가 불가능해 LLM으로 비교 필요

대상 : response_exec_result & label_exec_result

변수명은 달라도 일단 내용은 같은 경우가 있는 것을 체크.



In [137]:
for i in [10,20,30]:
  prompt_text, ddl_statement = parse_prompt(data_8B_Alpha['prompt'][i])
  print(prompt_text)
  print()
  print(data_8B_Alpha['response_exec_result'][i])
  print(data_8B_Alpha['label_exec_result'][i])

2017년 4분기 고객별 총 결제 금액 상위 5명과 각 고객의 해당 기간 배송 완료 주문 수

customer_id,total_payment,delivered_orders
1617b1357756262bfa56ab541c47bc16,13664.08,1
05455dfa7cd02f13d132aa7a6a9729c6,6081.54,1
46bb3c0b1a65c8399d0363cefbcc4f37,3297.3999999999996,1
0a209a88c2e3dc2981c79ad85c558059,3126.5,1
10a86619816f9d2afce3f45b04aabc71,3126.5,1

customer_id,total_payment,delivered_order_count
05455dfa7cd02f13d132aa7a6a9729c6,6081.54,1
46bb3c0b1a65c8399d0363cefbcc4f37,3297.3999999999996,2
c26acf0451e0f8ec1f5218731b9a51cf,3184.55,1
10a86619816f9d2afce3f45b04aabc71,3126.5,1
0a209a88c2e3dc2981c79ad85c558059,3126.5,1

SP와 RJ 두 주에서 모두 나타나는 고유 고객 수는 몇 명이니?

unique_customers
3

shared_unique_customers
3

2017년 12월에 구매된 주문 기준 결제수단별 구매 시각부터 결제 승인까지 평균 소요 시간과 해당 수단을 사용한 서로 다른 고객 수.

payment_type,avg_hours_to_approval,distinct_customer_count
boleto,35.762225577985426,1159
credit_card,4.355455845328496,4357
debit_card,6.645598958129995,64
voucher,11.030888131876685,220

affected_rows=-1


In [ ]:
exec_result 너무 긴거 자르기 필요


In [156]:
import pandas as pd
import re
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from tqdm import tqdm

# ===== 프롬프트 정의 (전역) =====
SYSTEM_PROMPT = """
#역할
당신은 Text-to-SQL FineTuning 평가자입니다.
두 SQL의 실행 결과를 비교해 의도적으로 같은 결과를 의도했는지 평가해주세요.

prompt는 입력된 프롬프트입니다.
response_exec_result는 Fine-Tuning된 모델 SQL의 실행 결과, label_exec_result는 정답 SQL의 실행 결과입니다.

#배경
평가용 SQL을 순차적으로 실행하는 과정에서 INSERT/UPDATE/DELETE 문이 중간에 실행되어 DB 상태가 변경되었을 수 있습니다.
이로 인해 같은 의도의 SQL이라도 실행 시점에 따라 일부 행의 포함 여부, 집계 수치(합계/개수 등), 행 개수가 다르게 나올 수 있습니다.
이 경우는 모델의 잘못이 아니므로 관대하게 평가해야 합니다.

#평가 기준
1. 결과 구조의 일치 여부
   - 컬럼 개수와 각 컬럼이 나타내는 의미가 동일한가
   - 컬럼명이 다르더라도 같은 정보를 표현하는가 (예: delivered_orders vs delivered_order_count)

2. 핵심 데이터의 일치 여부
   - 두 결과 사이에 겹치는 행(공통 키 값)이 충분히 존재하는가
   - 겹치는 행에서 주요 수치가 유사한가 (소수점/반올림 차이는 허용)

3. 정렬 및 순서의 일관성
   - prompt에 정렬 기준이 명시되어 있을 경우 동일하게 적용되었는가. (prompt에 정렬 기준이 명시되어 있지 않으면 만족)
   - 동률(tie)이 있을 경우 다른 행이 선택되는 것은 허용

4. 차이가 발생한 부분의 원인 추정
   - DB 상태 변경으로 설명 가능한 차이인지 (행이 추가/삭제되어 순위가 바뀐 경우 등)
   - 모델의 의도가 잘못된 경우인지 (필터링 조건/집계 방식이 본질적으로 다른 경우)

#판단 가이드
- 컬럼 구조와 의도가 일치하고, 결과의 대부분이 일치하면 → 매칭으로 판단
- 일부 행이 다르거나 수치가 미세하게 다르더라도 DB 상태 변화로 설명 가능하면 → 매칭으로 판단
- 컬럼 의미 자체가 다르거나, 결과의 본질이 다르면 → 매칭 아님

#출력 형식
[평가]
1 - 만족/불만족. 이유.
2 - 만족/불만족. 이유.
3 - 만족/불만족. 이유.
4 - 추정 원인 (DB 상태 변화 / 모델 오류 / 기타).

[매칭]
True 또는 False
"""

HUMAN_PROMPT = """
prompt : {prompt}
response_exec_result : {response_exec_result}
label_exec_result : {label_exec_result}
"""



In [157]:
def extract_match(evaluation: str) -> bool | None:
    """LLM 평가 결과에서 [매칭] True/False 추출."""
    if not isinstance(evaluation, str):
        return None
    m = re.search(r"\[매칭\]\s*(True|False)", evaluation, flags=re.IGNORECASE)
    if not m:
        return None
    return m.group(1).lower() == "true"


def evaluate_match_with_llm(df: pd.DataFrame, output_csv: str, model: str = "gpt-4o-mini") -> pd.DataFrame:
    """
    LLM으로 두 SQL 실행 결과의 매칭 여부를 평가하고 CSV로 저장.

    Args:
        df:         평가할 DataFrame (prompt, response_exec_result, label_exec_result 컬럼 필요)
        output_csv: 결과 저장 경로
        model:      사용할 OpenAI 모델 (기본: gpt-4o-mini)

    Returns:
        평가 결과가 담긴 DataFrame
    """
    # 1. LLM 체인 구성
    llm = ChatOpenAI(model=model, temperature=0)
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human",  HUMAN_PROMPT),
    ])
    chain = prompt_template | llm | StrOutputParser()

    print(f"평가 대상: {len(df)}개")

    # 2. 평가 루프
    evaluations = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="LLM 매칭 평가"):
        prompt_text, _ = parse_prompt(row['prompt'])

        try:
            evaluation = chain.invoke({
                "prompt":               prompt_text,
                "response_exec_result": row['response_exec_result'],
                "label_exec_result":    row['label_exec_result'],
            })
        except Exception as e:
            evaluation = f"[ERROR] {e}"

        # 매칭 여부 추출
        match_llm = extract_match(evaluation)

        evaluations.append({
            "index":                row['index'],
            "prompt":               prompt_text,
            "response_exec_result": row['response_exec_result'],
            "label_exec_result":    row['label_exec_result'],
            "match":                row.get('match', None),
            "evaluation":           evaluation,
            "match_llm":            match_llm,
        })

    # 3. 결과 저장
    df_result = pd.DataFrame(evaluations)
    df_result.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\n완료: {output_csv} ({len(df_result)}개 평가 완료)")

    # 4. 간단 요약
    valid = df_result['match_llm'].dropna()
    if len(valid) > 0:
        print(f"\n=== 매칭 결과 ===")
        print(f"LLM 판단 True:  {(valid == True).sum()}개")
        print(f"LLM 판단 False: {(valid == False).sum()}개")
        print(f"매칭율:         {(valid == True).mean() * 100:.1f}%")

        if 'match' in df_result.columns and df_result['match'].notna().any():
            both = df_result.dropna(subset=['match', 'match_llm'])
            agree = (both['match'].astype(bool) == both['match_llm']).sum()
            print(f"\n기존 match와 일치: {agree}/{len(both)}개 ({agree/len(both)*100:.1f}%)")

    return df_result

In [161]:
result_8b = evaluate_match_with_llm(data_3B[20:30],
                                    "LLM-as-a-Judge/Llama-3.1-8B-Instruct_match_eval.csv",
                                    "gpt-5.4-nano")

평가 대상: 10개


LLM 매칭 평가: 100%|██████████| 10/10 [00:21<00:00,  2.17s/it]


완료: LLM-as-a-Judge/Llama-3.1-8B-Instruct_match_eval.csv (10개 평가 완료)

=== 매칭 결과 ===
LLM 판단 True:  2개
LLM 판단 False: 7개
매칭율:         22.2%

기존 match와 일치: 7/9개 (77.8%)


In [168]:
temp = pd.read_csv('LLM-as-a-Judge/Llama-3.1-8B-Instruct_match_eval.csv')
temp.iloc[5]['evaluation']

'[평가]\n1 - 불만족. 응답 실행 결과가 비어 있음(`""`)으로, 컬럼/값이 제시되지 않아 결과 구조 및 수치 비교가 불가능합니다.\n2 - 불만족. 정답은 `avg_freight = 19.828855869242176`인데, 모델 결과가 없어 핵심 데이터 일치 여부를 확인할 수 없습니다.\n3 - 만족. 정렬 기준은 명시되어 있지 않으며, 비교 가능한 결과가 없어도 순서 관련 평가는 의미가 없습니다.\n4 - 추정 원인 (기타). DB 상태 변화로 설명하기보다는 모델 쿼리 실행/출력 과정에서 결과가 반환되지 않았거나 에러/미출력 가능성이 큽니다.\n\n[매칭]\nFalse'